# E11 · Combinación de lo que funcionó en E10
**Parte de:** E8b (bagging de 5 semillas, 5 modelos + LightGBM; CV 0,8179, público 0,80168).

En la ronda E10 se probaron 4 hipótesis en paralelo:

| Prueba | Resultado |
|---|---|
| E10a · redes neuronales desde cero | ✗ no aporta (stacking 0,8180) |
| **E10b · variables nuevas a partir del análisis de errores (H12-H17)** | ✓ **0,8299** (+1,2 pts; parte es sesgo de selección; la versión genérica da 0,8234) |
| E10c · otras formulaciones | LGBM-PS (LightGBM Powerset) suma +0,0025 sobre E8b |
| E10d · modelos de lenguaje por clase | ✗ no aporta |

**Objetivo.** Ver si LGBM-PS suma también sobre las variables de E10b, y generar los candidatos de envío.

### Glosario de esta prueba
| Término | Significado |
|---|---|
| **HS / TR / AG** | Las 3 etiquetas: discurso de odio / odio dirigido a una persona concreta / agresividad |
| **F1 medio** | F1 de la clase 1 de cada etiqueta, promediado entre HS, TR y AG (métrica principal) |
| **CV repetida** | Validación cruzada de 5 particiones repetida con 3 semillas (42, 43, 44); se reporta media ± desviación |
| **OOF** | *Out-of-fold*: predicción de cada fila del train hecha por un modelo que no la vio al entrenar |
| **Stacking** | Un metamodelo (regresión logística) aprende a combinar las probabilidades OOF de los modelos base |
| **H12-H17** | Variables nuevas de E10b: posesivo + insulto (*mi perra*), negación, *árabe* en estereotipo, tema migrante + deshumanización, demostrativo + grupo, demostrativo femenino (*esta puta*) |
| **LGBM-PS** | LightGBM multiclase sobre la combinación HS-TR-AG (Powerset) |
| **Versión genérica** | Las mismas hipótesis H12-H17 escritas sin palabras copiadas de tweets concretos del train |

El glosario completo está en `RESUMEN.md`.

In [1]:
import os, sys, platform
sys.path.insert(0, os.path.abspath('.'))          # el notebook vive en experimentos/, junto a toxic_base.py
import numpy as np, pandas as pd, sklearn, scipy
import toxic_base as tb
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 30); pd.set_option('display.max_colwidth', 120)
print('Python', platform.python_version(), '| numpy', np.__version__, '| pandas', pd.__version__,
      '| scikit-learn', sklearn.__version__, '| scipy', scipy.__version__)
print('Semillas de la CV repetida:', tb.SEEDS, '| particiones:', tb.N_FOLDS)
d = tb.Datos()
print('Datos en', d.data_dir, '| train', d.train.shape, '| test', d.test.shape)

Python 3.12.3 | numpy 2.5.4 | pandas 3.0.6 | scikit-learn 1.9.1 | scipy 1.18.1
Semillas de la CV repetida: (42, 43, 44) | particiones: 5


Datos en /home/jc/repositories/natural-language-processing-workshops/workshops/Kaggle-tweet-toxic/tweet-toxic-2026-puj | train (1731, 5) | test (1154, 2)


In [2]:
import ext_variables as ev, ext_formulaciones as ef
S5 = (42, 43, 44, 45, 46)
B6 = tb.BASES_ORIGINALES + ['LGBM']
DATOS = {'base': d, 'H12-H17': tb.Datos(extra_lex=[ev.variables()]),
         'H12-H17 genérica': tb.Datos(extra_lex=[ev.variables(generica=True)])}
for nombre, dd in DATOS.items():
    for s in S5:
        for b in B6 + ['LGBM-PS']:
            tb.oof(dd, b, s); tb.pred_test_folds(dd, b, s)
    tb.log('listo', nombre)
R = {}
for nombre, dd in DATOS.items():
    for bs_nom, bs in [('B6', B6), ('B6 + LGBM-PS', B6 + ['LGBM-PS'])]:
        R[f'{nombre} · {bs_nom}'] = tb.evaluar(dd, {s: tb.stacking_bagged(dd, bs, S5, s, con_test=False)[0] for s in tb.SEEDS})
tb.tabla(R)

[     1s] listo base


[   302s] listo H12-H17


[   582s] listo H12-H17 genérica


,F1 HS,F1 TR,F1 AG,F1 medio,±,EMR,F1 medio por semilla
Configuración,,,,,,,
base · B6,0.8071,0.8405,0.8062,0.8179,0.0012,0.7785,0.819 / 0.818 / 0.816
base · B6 + LGBM-PS,0.8123,0.8400,0.8088,0.8204,0.0004,0.7822,0.820 / 0.820 / 0.821
H12-H17 · B6,0.8211,0.8497,0.8188,0.8299,0.0014,0.7876,0.829 / 0.829 / 0.832
H12-H17 · B6 + LGBM-PS,0.8243,0.8475,0.8183,0.8300,0.0010,0.7909,0.829 / 0.831 / 0.831
H12-H17 genérica · B6,0.8153,0.8443,0.8105,0.8234,0.0017,0.7851,0.822 / 0.823 / 0.826
H12-H17 genérica · B6 + LGBM-PS,0.8174,0.8453,0.8105,0.8244,0.0015,0.7876,0.822 / 0.825 / 0.826


## Submissions

In [3]:
_, P8 = tb.stacking_bagged(d, B6, S5, 42); e8b = tb.binarizar(P8); e8b[d.err_excel_te] = 0
CAND = {'toxic_submission_E11a_variables_lgbmps.csv': ('H12-H17', B6 + ['LGBM-PS']),
        'toxic_submission_E11b_generica_lgbmps.csv': ('H12-H17 genérica', B6 + ['LGBM-PS']),
        'toxic_submission_E11c_base_lgbmps.csv': ('base', B6 + ['LGBM-PS'])}
for archivo, (dn, bs) in CAND.items():
    print('=' * 20, archivo)
    _, Pt = tb.stacking_bagged(DATOS[dn], bs, S5, 42)
    sub = tb.guardar_submission(DATOS[dn], tb.binarizar(Pt), archivo)
    print('Filas distintas frente a E8b (0,80168):', int((sub[tb.LABELS].values != e8b).any(1).sum()))

==================== toxic_submission_E11a_variables_lgbmps.csv
[   585s] Guardada /home/jc/repositories/natural-language-processing-workshops/workshops/Kaggle-tweet-toxic/experimentos/submissions/toxic_submission_E11a_variables_lgbmps.csv


Prevalencia train : {'HS': np.float64(0.437), 'TR': np.float64(0.3), 'AG': np.float64(0.382)}
Prevalencia pred. : {'HS': np.float64(0.406), 'TR': np.float64(0.302), 'AG': np.float64(0.371)}
Cambios frente a la base 0,79160: 82 filas ({'HS': 58, 'TR': 27, 'AG': 41})
Filas distintas frente a E8b (0,80168): 52
==================== toxic_submission_E11b_generica_lgbmps.csv


[   585s] Guardada /home/jc/repositories/natural-language-processing-workshops/workshops/Kaggle-tweet-toxic/experimentos/submissions/toxic_submission_E11b_generica_lgbmps.csv


Prevalencia train : {'HS': np.float64(0.437), 'TR': np.float64(0.3), 'AG': np.float64(0.382)}
Prevalencia pred. : {'HS': np.float64(0.406), 'TR': np.float64(0.304), 'AG': np.float64(0.373)}
Cambios frente a la base 0,79160: 79 filas ({'HS': 55, 'TR': 25, 'AG': 38})
Filas distintas frente a E8b (0,80168): 46
==================== toxic_submission_E11c_base_lgbmps.csv


[   585s] Guardada /home/jc/repositories/natural-language-processing-workshops/workshops/Kaggle-tweet-toxic/experimentos/submissions/toxic_submission_E11c_base_lgbmps.csv


Prevalencia train : {'HS': np.float64(0.437), 'TR': np.float64(0.3), 'AG': np.float64(0.382)}
Prevalencia pred. : {'HS': np.float64(0.408), 'TR': np.float64(0.305), 'AG': np.float64(0.368)}
Cambios frente a la base 0,79160: 65 filas ({'HS': 45, 'TR': 18, 'AG': 24})
Filas distintas frente a E8b (0,80168): 25


## Conclusión
- Las variables H12-H17 son la mejora grande: **0,8179 → 0,8299**, en las 3 semillas. LGBM-PS suma sobre la base (+0,0025), pero sobre H12-H17 ya no (0,8300): aportan señal parecida.
- La versión genérica (sin palabras copiadas del train) da 0,8234-0,8244. Es la cota más conservadora de la mejora, porque las H12-H17 se escribieron mirando errores del mismo train con que se evalúan.
- Candidatos: **E11a** (H12-H17 + LGBM-PS, CV 0,8300), **E11b** (genérica + LGBM-PS, 0,8244) y **E11c** (sin variables nuevas, solo LGBM-PS, 0,8204).